# Tutorial 11: Advanced Techniques

**Prerequisites:** All previous tutorials (T00–T10)

**What you'll learn:**
- **Linear probes**: Training classifiers on internal representations
- **Activation steering**: Adding directions to control model behavior
- **Gradient-based attribution**: Using gradients to measure importance
- **Where to go from here**: The full IRTK toolkit

---

This tutorial introduces three additional powerful techniques and points you toward the full breadth of IRTK's 157+ analysis modules.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig

cfg = HookedTransformerConfig(
    n_layers=2, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
print('Model ready')

## Technique 1: Linear Probes

**Linear probes** test whether specific information is **linearly represented** in the residual stream. You train a small linear classifier on top of activations to predict some property:

- "Is this token a noun?" → probe on residual stream at that position
- "Has the model seen a repeated token?" → probe on later layers
- "Is the subject singular or plural?" → probe at the verb position

If a linear probe succeeds, the information is **linearly accessible** at that point — meaning downstream components could read it with a simple linear operation.

In [ ]:
from irtk.probes import LinearProbe, train_linear_probe

# Task: Can the model's residual stream distinguish even vs odd token IDs?
# (A toy task, but illustrates the technique)

rng = jax.random.PRNGKey(0)
hook_name = 'blocks.1.hook_resid_post'  # After the last layer

# Collect activations and labels
activations = []
labels = []

for i in range(200):
    rng, subkey = jax.random.split(rng)
    tokens = jax.random.randint(subkey, (5,), 0, cfg.d_vocab)
    _, cache = model.run_with_cache(tokens)
    
    # Use the last position's activation
    act = np.array(cache[hook_name][-1])  # [d_model]
    label = int(tokens[-1]) % 2  # 0 = even, 1 = odd
    
    activations.append(act)
    labels.append(label)

X = jnp.array(np.stack(activations))  # [n_examples, d_model]
y = jnp.array(labels)                 # [n_examples]

# Train a linear probe
probe, accuracy = train_linear_probe(
    X, y, n_classes=2, n_epochs=100, lr=1e-2,
)

print(f'Linear probe for even/odd token ID:')
print(f'  Training accuracy: {accuracy:.1%}')
print(f'  Chance level:      50.0%')
if accuracy > 0.7:
    print(f'  The model\'s residual stream encodes even/odd information!')
else:
    print(f'  The model does not linearly encode even/odd (expected with random weights).')

In [ ]:
# Probe at different layers to see where information appears
print('Probing accuracy by layer:\n')

for layer in range(cfg.n_layers):
    for stage in ['hook_resid_pre', 'hook_resid_post']:
        hook = f'blocks.{layer}.{stage}'
        
        acts = []
        lbls = []
        rng2 = jax.random.PRNGKey(layer * 10 + (0 if 'pre' in stage else 1))
        
        for i in range(200):
            rng2, subkey = jax.random.split(rng2)
            tokens = jax.random.randint(subkey, (5,), 0, cfg.d_vocab)
            _, cache = model.run_with_cache(tokens)
            acts.append(np.array(cache[hook][-1]))
            lbls.append(int(tokens[-1]) % 2)
        
        X2 = jnp.array(np.stack(acts))
        y2 = jnp.array(lbls)
        _, acc = train_linear_probe(X2, y2, n_classes=2, n_epochs=100, lr=1e-2)
        
        name = stage.replace('hook_resid_', '')
        bar = '#' * int(acc * 20)
        print(f'  L{layer} {name:4s}: {acc:.1%} {bar}')

print(f'\nIf accuracy increases through layers, the model is building up this representation.')

## Technique 2: Activation Steering

**Activation steering** (also called "representation engineering" or "activation addition") modifies the model's behavior by adding a direction vector to the residual stream during inference.

The idea: if you can find a direction in activation space that corresponds to a concept (e.g., "positive sentiment"), you can add or subtract it to steer the model's output.

```
steered_residual = original_residual + α * steering_direction
```

In [ ]:
from irtk.steering import compute_steering_vector, add_vector

# Compute a steering vector: the difference in residual stream between
# two sets of inputs (contrastive pair approach)

# "Positive" inputs (high token IDs) vs "negative" inputs (low token IDs)
positive_tokens = [jnp.array([80, 85, 90, 95, 99]) for _ in range(10)]
negative_tokens = [jnp.array([1, 5, 10, 15, 20]) for _ in range(10)]

steering_vector = compute_steering_vector(
    model, positive_tokens, negative_tokens,
    hook_name='blocks.0.hook_resid_post',
    pos=-1,
)

print(f'Steering vector shape: {steering_vector.shape}')
print(f'Steering vector norm: {float(jnp.linalg.norm(steering_vector)):.4f}')

In [ ]:
# Apply the steering vector and see how predictions change
test_tokens = jnp.array([40, 45, 50, 55, 60])
logits_base = model(test_tokens)

print(f'Effect of steering at different strengths:\n')
base_pred = int(jnp.argmax(logits_base[-1]))
print(f'  No steering:    top prediction = token {base_pred}')

for alpha in [0.5, 1.0, 2.0, 5.0]:
    logits_steered = add_vector(
        model, test_tokens, steering_vector,
        hook_name='blocks.0.hook_resid_post',
        alpha=alpha,
    )
    pred = int(jnp.argmax(logits_steered[-1]))
    diff = float(jnp.linalg.norm(logits_steered[-1] - logits_base[-1]))
    print(f'  α={alpha:4.1f}: top prediction = token {pred:3d}, '
          f'logit change = {diff:.4f}')

print(f'\nSteering shifts the prediction! Larger α = stronger effect.')

## Technique 3: Gradient-Based Attribution

**Gradient-based methods** measure how much each input feature contributes to the output by computing gradients:

- **Gradient × Input**: `∂output/∂input * input` — fast, one-pass
- **Integrated Gradients**: Average gradients along a path from baseline to input — more principled

These give a complementary view to activation patching.

In [ ]:
from irtk.gradients import grad_times_input, integrated_gradients

tokens = jnp.array([10, 42, 17, 88, 55])
target_token = int(jnp.argmax(model(tokens)[-1]))

# Gradient × input: how does each embedding dimension contribute?
gxi = grad_times_input(model, tokens, target_token=target_token, pos=-1)
print(f'Gradient × Input attribution:')
print(f'  Shape: {gxi.shape}  [seq, d_model]')
print(f'  (Attribution of the logit for token {target_token} to each input dimension)')

# Sum across d_model to get per-position importance
pos_importance = np.array(jnp.sum(jnp.abs(gxi), axis=-1))  # [seq]
print(f'\nPer-position importance:')
for p in range(len(tokens)):
    bar = '#' * int(pos_importance[p] * 50)
    print(f'  Position {p} (token {int(tokens[p]):2d}): {pos_importance[p]:.4f} {bar}')

In [ ]:
# Integrated gradients: more principled attribution
ig = integrated_gradients(
    model, tokens, target_token=target_token, pos=-1, n_steps=20,
)

ig_importance = np.array(jnp.sum(jnp.abs(ig), axis=-1))
print(f'Integrated gradients per-position importance:')
for p in range(len(tokens)):
    bar = '#' * int(ig_importance[p] * 50)
    print(f'  Position {p} (token {int(tokens[p]):2d}): {ig_importance[p]:.4f} {bar}')

print(f'\nCompare with gradient × input — they often agree on which')
print(f'positions matter most, but integrated gradients satisfies')
print(f'stronger theoretical guarantees (completeness axiom).')

## Where to Go From Here

You now have a solid foundation in mechanistic interpretability! Here's how to continue:

### Immediate Next Steps

1. **Try with a real model**: Load GPT-2 with `HookedTransformer.from_pretrained('gpt2')` and apply these techniques to actual language
2. **Reproduce a classic result**: The induction heads analysis (see `notebooks/induction_heads.ipynb`)
3. **Browse the API cheatsheet**: `notebooks/01_api_cheatsheet.ipynb` covers all 157+ modules

### IRTK's Full Toolkit (157+ Modules)

The tutorials have covered the core techniques. IRTK also includes:

| Category | What's available |
|----------|------------------|
| **Circuit Analysis** | OV/QK circuits, composition scores, direct logit attribution, path expansion, virtual weights |
| **Patching Variants** | Denoising/noising, mean/resample ablation, causal mediation, patchscopes |
| **Feature Analysis** | SAEs, cross-coders, transcoders, feature geometry, feature interaction maps |
| **Probing** | Linear/nonlinear probes, sparse probes, concept directions, LEACE erasure |
| **Attribution** | Shapley values, path attribution, gradient flow, token-to-neuron maps |
| **Model Surgery** | Head/MLP transplant, weight knockout, activation surgery, attention forcing |
| **Representation** | Manifold geometry, CKA alignment, bottleneck analysis, information flow |
| **Safety** | Refusal directions, deception detection, trojan signatures, memorization detection |
| **Training Dynamics** | Phase transitions, grokking detection, circuit formation, developmental interp |
| **Benchmarks** | Faithfulness correlation, logit diff, KL divergence, ablation effect size |

Each module has its own notebook in the `notebooks/` directory (numbered 00–146).

In [ ]:
# Quick tour of a few more tools
from irtk.circuits import direct_logit_attribution

# Direct logit attribution: which heads contribute most to the prediction?
tokens = jnp.array([1, 42, 17, 88, 55])
dla = direct_logit_attribution(model, tokens, pos=-1)

print('Direct logit attribution (per head):')
for layer in range(cfg.n_layers):
    for head in range(cfg.n_heads):
        val = float(dla[layer, head])
        bar = '+' * int(max(0, val) * 5) + '-' * int(max(0, -val) * 5)
        print(f'  L{layer}H{head}: {val:+.4f} {bar}')

In [ ]:
from irtk.attention_utils import attention_entropy

# How focused/diffuse is each head's attention?
_, cache = model.run_with_cache(tokens)

print('Attention entropy (low = focused, high = diffuse):')
for layer in range(cfg.n_layers):
    patterns = cache[f'blocks.{layer}.attn.hook_pattern']
    for head in range(cfg.n_heads):
        ent = attention_entropy(patterns[head])
        mean_ent = float(jnp.mean(ent))
        max_ent = float(jnp.log(jnp.array(len(tokens))))
        bar = '#' * int(mean_ent / max_ent * 15)
        print(f'  L{layer}H{head}: {mean_ent:.3f} / {max_ent:.2f} {bar}')

## Summary: The Complete Tutorial Series

You've completed all 12 tutorials! Here's what you've learned:

| Tutorial | Core Skill |
|----------|------------|
| **T00** | What mechinterp is and why it matters |
| **T01** | How transformers work, component by component |
| **T02** | How to observe and intervene using hooks |
| **T03** | The residual stream: decomposition and attribution |
| **T04** | Attention heads: QK/OV circuits, head types, composition |
| **T05** | MLPs: neuron analysis, features, polysemanticity |
| **T06** | The complete investigation workflow |
| **T07** | Logit lens: watching predictions form |
| **T08** | Activation patching: causal hypothesis testing |
| **T09** | Circuit discovery: finding minimal circuits |
| **T10** | Sparse autoencoders: feature disentanglement |
| **T11** | Linear probes, steering, gradients, and the full toolkit |

### Recommended Reading

- [A Mathematical Framework for Transformer Circuits](https://transformer-circuits.pub/2021/framework/index.html) (Anthropic, 2021)
- [In-context Learning and Induction Heads](https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html) (Anthropic, 2022)
- [Towards Monosemanticity](https://transformer-circuits.pub/2023/monosemantic-features/) (Anthropic, 2023)
- [Scaling Monosemanticity](https://transformer-circuits.pub/2024/scaling-monosemanticity/) (Anthropic, 2024)
- [200 Concrete Problems in Interpretability](https://docs.google.com/spreadsheets/d/1oOdrQ80jDK-aGn-EVdDt3dg65GhmzrvBWzJ6MUZB8n4/) (Neel Nanda)

Happy investigating!